In [1]:
import pandas as pd
import numpy as np

In [2]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [93]:
universal_dataset = pd.read_parquet(
    "../data/processed/universal_dataset.parquet",
    engine="pyarrow"
)

In [94]:
df = pd.DataFrame(universal_dataset)
print(df.shape)
print(df.columns)
df.sample(5)

(100000, 11)
Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase', 'category'],
      dtype='str')


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
17355,4.0,Very enjoyable,Jaimoe with his solo band is not playing the ...,[],B007KBRKAA,B007KBRKAA,AHQRGKJCIHJKSD7JVGUL3BXQQNDA,1541473448331,0,False,Digital_Music
547,5.0,A marvelous disc in every respect,"This 1986 recording, dedicated to the later or...",[],B003TZOT50,B003TZOT50,AFEZ5USUDVJLGSOB5SVVBUOLVNAQ,1605578203250,0,False,Digital_Music
44793,5.0,TP-Link AC1200 Wireless Wi-Fi Gigabit Router (...,Flawless performance for the past 5 years. 2 p...,[],B00JZFG6QS,B00JZFG6QS,AFWVN52MRBWOTIK7UGXBWGOY4HBA,1556989303080,0,False,Electronics
57290,5.0,You can get lost in the leaves.,[[VIDEOID:ef71921646e193ea1271924f22bf17c2]] I...,[{'small_image_url': 'https://m.media-amazon.c...,B0B6Z2Y57Q,B0B6Z2Y57Q,AFHPJTBQYQHQXIFBAODXDAIGCGMQ,1664323314262,0,False,Electronics
46132,5.0,"Powerful, flashy, and controllable.",This isn't the first time I've installed memor...,[],B07D1XJWSJ,B07DKMJ61N,AEUAL3EJKUSTNB4YY6STLYGTJALA,1544296047046,0,False,Electronics


In [95]:
df.isnull().sum()

rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
category             0
dtype: int64

In [96]:
df.describe()

,rating,timestamp,helpful_vote
count,100000.00000,1.000000e+05,100000.000000
mean,4.41173,1.548512e+12,1.230820
std,1.13432,9.700586e+10,9.221351
min,1.00000,9.049815e+11,0.000000
25%,4.00000,1.490279e+12,0.000000
50%,5.00000,1.571017e+12,0.000000
75%,5.00000,1.620922e+12,1.000000
max,5.00000,1.679872e+12,777.000000


In [97]:
#Exclude columns with unhashable list types (like 'images') to check for duplicates
df.drop(columns = ['images']).duplicated().sum()

np.int64(221)

In [98]:
df[df.duplicated(subset = ["text", "user_id", "parent_asin"], keep = False )]

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
1273,5.0,Piano Solo,I am singing solo for Christmas Cantata...this...,[],B0012A4SFA,B0012A4SFA,AEQ4BRZQAEE7UKKOPVTM2EQVGV3A,1384398192000,1,True,Digital_Music
1274,5.0,Piano Solo,I am singing solo for Christmas Cantata...this...,[],B0012A4SFA,B0012A4SFA,AEQ4BRZQAEE7UKKOPVTM2EQVGV3A,1384398192000,1,True,Digital_Music
1340,5.0,great price,great product. great price,[],B002LVGU82,B002LVGU82,AHKQKE46BOZQDE6TIKGSOUS3NEYQ,1548159682656,0,True,Digital_Music
1341,5.0,great price,great product. great price,[],B002LVGU82,B002LVGU82,AHKQKE46BOZQDE6TIKGSOUS3NEYQ,1548159682656,0,True,Digital_Music
1427,5.0,Scurve E. Dogs...,"Scuttle me bones, me buckoes, 'ere be more sal...",[],B00A82J4X4,B00A82J4X4,AEK27DQ54G7PIWW627UHT72TFX4A,1359670654000,1,False,Digital_Music
...,...,...,...,...,...,...,...,...,...,...,...
98674,5.0,Great item,Started working on a go bag. Holds more than y...,[],B01LW81EBO,B07VX81L84,AHH645F7BT3BTF7YCMFD5W4EXTGQ,1610835944368,0,True,Health_and_Personal_Care
98816,5.0,Great,"Strong vinegar, delisted and using as househol...",[],B08NPN1PG3,B0BR8WTXV5,AFPQRHHT4J5JAKKG4HGRCQNMDPIQ,1617632842246,2,True,Health_and_Personal_Care
98820,5.0,Great,"Strong vinegar, delisted and using as househol...",[],B08NPN1PG3,B0BR8WTXV5,AFPQRHHT4J5JAKKG4HGRCQNMDPIQ,1617632842246,2,True,Health_and_Personal_Care
99155,3.0,A bit disappointing,"Meh. Plates are flimsy, utensils are tiny, and...",[],B07VTNLRJC,B07VTNLRJC,AE2VA3TLLKW33LOGI6KMXC6AROUA,1583358635711,0,True,Health_and_Personal_Care


In [99]:
df = df.drop_duplicates(subset = ["text", "user_id", "parent_asin"],keep = False)

# Feature Engineering

In [100]:
#1. Remove HTML Tags
def remove_html(tags):
    import re
    clean = re.compile('<.*?.')
    return re.sub(clean, '', tags)
df['text'] = df['text'].apply(remove_html)

In [101]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [102]:
#2. Convert everything to lower
def lower_case(text):
    return text.lower()
df['text'] = df['text'].apply(lower_case)

In [103]:
df['text'][20]

'the follow-up to his "the street was always there" collection, and basically, it\'s more of the same.  a journeyman\'s collection of folk and folk-rock covers (mostly) delivered in andersen\'s smooth, thoughtful style.  if you were a fan of \'street,\' then "waves" will be right in your wheelhouse.  three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  i much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [104]:
#Removing special char
def remove_special(text):
    if not isinstance(text, str):
        return ""

    x = ""
    for i in text:
        if i.isalnum():
            x += i
        else:
            x += " "

    return x

In [105]:
df['text'] = df['text'].apply(remove_special)

In [107]:
df['text'][20]

'the follow up to his  the street was always there  collection  and basically  it s more of the same   a journeyman s collection of folk and folk rock covers  mostly  delivered in andersen s smooth  thoughtful style   if you were a fan of  street   then  waves  will be right in your wheelhouse   three and one half stars and a mildly enthusiastic recommendation from this longtime fan   i much prefer him in singer   songwriter mode rather than as an interpreter of other writers  works '

In [108]:
#Remove stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ujjwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [109]:
from nltk.corpus import stopwords

In [110]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    x = []

    for i in text.split():
        if i not in stop_words:
            x.append(i)

    return " ".join(x)

In [111]:
df['text']= df['text'].apply(remove_stopwords)

In [112]:
df['text'][20]

'follow street always collection basically journeyman collection folk folk rock covers mostly delivered andersen smooth thoughtful style fan street waves right wheelhouse three one half stars mildly enthusiastic recommendation longtime fan much prefer singer songwriter mode rather interpreter writers works'

In [113]:
#Perform stemming
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [135]:
def stem_words(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [136]:
df['text'] = df['text'].apply(stem_words)

In [138]:
df['text'][20]

'follow street alway collect basic journeyman collect folk folk rock cover mostli deliv andersen smooth thought style fan street wave right wheelhous three one half star mildli enthusiast recommend longtim fan much prefer singer songwrit mode rather interpret writer work'

In [139]:
def join_back(text):
    return "".join(text)

In [140]:
join_back(['d', 'o', 'l', 'l', 'a', 'r', ' ', 'm', 'a', 'n', 'y'])

'dollar many'

In [141]:
df['text'] = df['text'].apply(join_back)

In [142]:
df['text'][20]

'follow street alway collect basic journeyman collect folk folk rock cover mostli deliv andersen smooth thought style fan street wave right wheelhous three one half star mildli enthusiast recommend longtim fan much prefer singer songwrit mode rather interpret writer work'

In [153]:
def clean_text(text):
    text = remove_html(text)
    text = lower_case(text)
    text = remove_special(text)
    text = remove_stopwords(text)
    text = stem_words(text)
    text = join_back(text)

    return text

# Data preprocessing and transformation

In [166]:
#Create sentiment labels from ratings
def get_sentiment(rating):
    if rating < 3:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df['sentiment'] = df['rating'].apply(get_sentiment)

In [167]:
#Testinf sentiment labeling
df['sentiment'][20]

'positive'

In [221]:
%pip install imbalanced-learn


   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [246]:
def preprocess_text(x):
    return [clean_text(text) for text in x]

In [247]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB

pipe = Pipeline([
    ("clean_text", FunctionTransformer(
        preprocess_text,
        validate=False
    )),

    ("vectorizer", TfidfVectorizer(max_features=10000)),

    ("sampler", RandomOverSampler(
        sampling_strategy={
            "negative": 33506,
            "neutral": 33506
        },
        random_state=42
    )),

    ("classifier", ComplementNB())
])

In [248]:
#Input features and target
X = df['text']
y = df["rating"].apply(get_sentiment)

In [249]:
#Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42,stratify=y)

In [250]:
print("training_shape", X_train.shape)
print("testing_shape",X_test.shape)

training_shape (79567,)
testing_shape (19892,)


In [251]:
pipe.fit(X_train, y_train)

,steps,"[('clean_text', ...), ('vectorizer', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[<U8](3,)","['negative','neutral','positive']"
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function pre...001F91156B1A0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True


In [252]:
y_pred = pipe.predict(X_test)

In [253]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.5, 1.0, 2.0]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Macro F1:", grid.best_score_)

Best Parameters: {'classifier__alpha': 2.0}
Best CV Macro F1: 0.5325476974000999


In [255]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7716167303438568

Classification Report:
              precision    recall  f1-score   support

    negative       0.40      0.67      0.50      1870
     neutral       0.19      0.38      0.25      1262
    positive       0.96      0.81      0.88     16760

    accuracy                           0.77     19892
   macro avg       0.52      0.62      0.55     19892
weighted avg       0.86      0.77      0.80     19892


Confusion Matrix:
[[ 1255   339   276]
 [  430   482   350]
 [ 1429  1719 13612]]


In [256]:
import os

os.makedirs("../models", exist_ok=True)

In [257]:
import joblib

joblib.dump(best_model, "../models/model.pkl")

print("Model saved successfully!")

Model saved successfully!
